In [1]:
# Importing libraries & frameworks 
import genesis as gs
import numpy as np
import cv2
from pathlib import Path


[I 04/15/25 18:42:50.941 4018786] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    precision='32',
    debug=False,
    eps=1e-12,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=False, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())

# Bottle model
plastic_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="../bottles/plastic_bottle/bottle.stl",  # Absolute or relative path
        pos=(0.3, 0.0, 0.1),            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.0002                      
    )
)


# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = (0.0, 0.0, 0.09),
    res    = (1280, 960),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)

# Building the Scene
scene.build()


[Genesis] [18:42:53] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [18:42:53] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [18:42:53] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [18:42:54] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [18:42:54] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [18:42:54] [INFO] Scene <7da4cfc> created.
[Genesis] [18:42:54] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <0bfba71>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [18:42:54] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <f10f267>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/spot_gen/Exp_RigidEntity/bottles/plastic_bottle/bottle.stl')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [18:42:55] [INFO] Building scene <7da4cfc>...
[Genesis] [18:42:59] [INFO] Compiling simulation kernels...
[Genesis] [18:43:02] [INFO] Building visualizer...


In [3]:
# Output directory
output_dir = Path("../water_bottle_dataset/")
output_dir.mkdir(parents=True, exist_ok=True)

# Number of samples
num_samples = 1  
data_metadata = []

In [ ]:
# SIDE VIEW: Getting RGB, Depth, Segmentation data's & Normal Map 
for i in range(num_samples):
    cam.set_pose(
    pos    = (1, 0, 0.1),
    lookat = (0, 0, 0.09),
    up = (0, 0, 1),
    )

    # Capture RGB-D data
    side_render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False)
    side_rgb, side_depth, side_seg, side_normal_map = side_render_output
    
    # Save data
    view_id = f"side_view_{i:02d}"
    np.save(output_dir / f"side_rgb.npy", side_rgb)
    np.save(output_dir / f"side_depth.npy", side_depth)
    np.save(output_dir / f"side_segmentation_mask.npy", side_seg)
    np.save(output_dir / f"side_normal_map.npy", side_normal_map)
    
    # Save RGB as PNG for visual verification
    cv2.imwrite(str(output_dir / "side_rgb.png"), cv2.cvtColor(side_rgb, cv2.COLOR_RGB2BGR))
    
    # Save depth as PNG (scaled to 0-255)
    side_depth_min, side_depth_max = np.min(side_depth), np.max(side_depth)
    if side_depth_max > side_depth_min:  # Avoid division by zero
        side_depth_scaled = (side_depth - side_depth_min) / (side_depth_max - side_depth_min) * 255
    else:
        side_depth_scaled = np.zeros_like(side_depth)
    
    side_depth_scaled = side_depth_scaled.astype(np.uint8)
    cv2.imwrite(str(output_dir / f"depth_{view_id}.png"), side_depth_scaled)
    